In [3]:
# pip install torch torchvision torchaudio

In [6]:
# pip install onnxscript

In [9]:
import torch
import torch.nn as nn
import torch.export
import warnings

# Optional: Suppress the internal library FutureWarning from miniconda
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Define the model
# "SimpleNet"—a tiny brain that takes 10 numbers and spits out 1 answer.
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc = nn.Linear(10, 1)

    def forward(self, x):
        return self.fc(x)

# 2. Initialize model and dummy input
model = SimpleNet()
model.eval()
dummy_input = torch.randn(1, 10)

# 3. Define dynamic shapes (Replacing the old dynamic_axes)
# This tells the exporter that the 0th dimension of 'input' is dynamic
batch_dim = torch.export.Dim("batch", min=1)
dynamic_shapes = {"x": {0: batch_dim}}

# 4. Export to ONNX ("Universal Translator" to create a file called model.onnx)
# Note: use dynamo=True for the modern export engine
onnx_program = torch.onnx.export(
    model,
    (dummy_input,),
    "model.onnx",
    export_params=True,
    opset_version=18,
    input_names=['input'],
    output_names=['output'],
    dynamic_shapes=dynamic_shapes,
    dynamo=True
)

print("Model successfully exported to model.onnx with Opset 18")

[torch.onnx] Obtain model graph for `SimpleNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model successfully exported to model.onnx with Opset 18


In porduction, there is no need of PyTorch code; lightweight ONNX Runtime (ORT)is enough 

In [12]:
# pip install onnxruntime

In [14]:
import onnxruntime as ort
import numpy as np

# 1. Start an inference session (this loads the model)
# You can specify "providers" like CUDA (GPU) or CPU
session = ort.InferenceSession("model.onnx", providers=['CPUExecutionProvider'])

# 2. Prepare your input data as a standard NumPy array
input_data = np.random.randn(1, 10).astype(np.float32)

# 3. Run the model
# The first argument is the output name, the second is a dictionary of inputs
outputs = session.run(['output'], {'input': input_data})

print("Model Prediction:", outputs[0])

Model Prediction: [[0.0843983]]


### The notebook shows a complete, working pipeline where an AI "brain," is built, translated it into a universal format, and then used it to make a prediction.

### 1. The Big Picture: the notebook performed a "conversion and inference" workflow. Think of it as building a recipe in a specialized kitchen (PyTorch) and then vacuum-sealing it (ONNX) so it can be cooked anywhere in the world (ONNX Runtime).

### The Steps in the Code:
* **The Model (SimpleNet):** A tiny neural network was created. It has one layer that takes **10 input numbers** and performs a mathematical calculation to produce **1 output number**.
* **The Export:** The model was saved as `model.onnx`. This file contains the "frozen" math of the network. It no longer needs PyTorch to function.
* **The Result:** 10 random numbers were fed into that `.onnx` file, and it gave you a prediction.

---

## 2. What does the result mean?
`Model Prediction: [[0.0843983]]`

### What is this number?
This is the **raw output** of your neural network. 
* Because you used `np.random.randn(1, 10)`, you gave the model 10 random pieces of data.
* The model took those 10 numbers, multiplied them by its internal "weights," added a "bias," and spat out **0.0843983**.

**In a real-world scenario:**
* If this were a **House Price Predictor**, that number might represent $84,398.
* If this were a **Sentiment Analyzer**, a positive number might mean the model thinks a text is "Happy."
* In your case, since the model isn't "trained" on real data yet, it's just doing random math—but it proves that the **plumbing is working**.

---

### 3. Why is this a success?
The fact that there is a numerical result means:
1.  **The Translation worked:** The complex PyTorch code was successfully simplified into ONNX format.
2.  **The Environment is ready:** There are the right libraries (`torch`, `onnxscript`, `onnxruntime`, and `numpy`) talking to each other correctly.
3.  **The Data Flow is correct:** The data is successfully passed from a NumPy array (standard Python math) into the ONNX engine and got a result back.


Now that the "bridge" is built, and the `SimpleNet` can be replaced with a much more powerful model 